In [1]:
import sunpy
import sunpy.map
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d,RegularGridInterpolator
from astropy.coordinates import SkyCoord
from scipy.io import readsav
from matplotlib.patches import Rectangle
from scipy.ndimage import zoom
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord
import cv2
import glob
import os

In [ ]:
#检索目录
hmi_path=r'C:\Learning\PHD2nd\sunspotscar\data\X'
file_name=os.listdir(hmi_path)
file_path=[]
for i in file_name:
    file_path.append(os.path.join(hmi_path,i,'hmi.B_720s'))

In [27]:
print(file_path[0])

C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\hmi.B_720s


In [31]:
print(glob.glob(os.path.join(hmi_path, '*.field.fits')))

['C:\\Learning\\PHD2nd\\sunspotscar\\data\\X\\20120712_153700_UTC\\hmi.B_720s\\hmi.b_720s.20120712_153600_TAI.field.fits']


In [ ]:
#计算生成Br.fits
cnt=0
for i in file_path:
    cnt+=1
    print(cnt)
    d=readsav(os.path.join(i, 'bxyz.sav'))
    mapbx = d["mapbx"][0][0]
    mapby = d["mapby"][0][0]
    mapbz = d["mapbz"][0][0]
    hmi_path=i
    B_map_4098 = sunpy.map.Map(glob.glob(os.path.join(hmi_path, '*.field.fits'))[0]).rotate()
    B_map_data=B_map_4098.data
    B_map_data=cv2.resize(B_map_data,(4096,4096),interpolation=cv2.INTER_AREA)
    B_map=sunpy.map.Map(B_map_data,B_map_4098.meta)
    inc_map_4098 = sunpy.map.Map(glob.glob(os.path.join(hmi_path, '*.inclination.fits'))[0]).rotate()
    inc_map_data=inc_map_4098.data
    inc_map_data=cv2.resize(inc_map_data,(4096,4096),interpolation=cv2.INTER_AREA)
    inc_map=sunpy.map.Map(inc_map_data,inc_map_4098.meta)
    azi_map_4098 = sunpy.map.Map(glob.glob(os.path.join(hmi_path, '*.azimuth.fits'))[0]).rotate()
    azi_map_data=azi_map_4098.data
    azi_map_data=cv2.resize(azi_map_data,(4096,4096),interpolation=cv2.INTER_AREA)
    azi_map=sunpy.map.Map(azi_map_data,azi_map_4098.meta)
    # 1 获取 header 参数
    phi0 = B_map.observer_coordinate.lon.to(u.rad).value
    b    = B_map.observer_coordinate.lat.to(u.rad).value
    # WCS rotation matrix
    pc = B_map.wcs.wcs.pc

    # P-angle（弧度）
    p = np.arctan2(pc[0,1], pc[0,0])

    # 2 像素坐标
    ny,nx = B_map.data.shape
    y,x = np.mgrid[0:ny,0:nx]

    # 3 skycoord
    coords = pixel_to_skycoord(x,y,B_map.wcs)

    # 4 heliographic
    hg = coords.transform_to(frames.HeliographicStonyhurst)

    phi = hg.lon.to(u.rad).value
    lam = hg.lat.to(u.rad).value
    #计算矩阵参数
    dphi = phi - phi0

    k11 = np.cos(lam)*(np.sin(b)*np.sin(p)*np.cos(dphi) + np.cos(p)*np.sin(dphi)) \
        - np.sin(lam)*(np.cos(b)*np.sin(p))

    k12 = -np.cos(lam)*(np.sin(b)*np.cos(p)*np.cos(dphi) - np.sin(p)*np.sin(dphi)) \
        + np.sin(lam)*(np.cos(b)*np.cos(p))

    k13 = np.cos(lam)*np.cos(b)*np.cos(dphi) + np.sin(lam)*np.sin(b)


    k21 = np.sin(lam)*(np.sin(b)*np.sin(p)*np.cos(dphi) + np.cos(p)*np.sin(dphi)) \
        + np.cos(lam)*(np.cos(b)*np.sin(p))

    k22 = -np.sin(lam)*(np.sin(b)*np.cos(p)*np.cos(dphi) - np.sin(p)*np.sin(dphi)) \
        - np.cos(lam)*(np.cos(b)*np.cos(p))

    k23 = np.sin(lam)*np.cos(b)*np.cos(dphi) - np.cos(lam)*np.sin(b)


    k31 = -np.sin(b)*np.sin(p)*np.sin(dphi) + np.cos(p)*np.cos(dphi)

    k32 =  np.sin(b)*np.cos(p)*np.sin(dphi) + np.sin(p)*np.cos(dphi)

    k33 = -np.cos(b)*np.sin(dphi)
    #这个Bx,By,Bz应该就是mapbx,mapby,mapbz，不知道为什么丢了一块
    Bx,By,Bz=mapbx,mapby,mapbz
    Br =(k11*Bx + k12*By + k13*Bz)
    Btheta = k21*Bx + k22*By + k23*Bz
    Bphi = k31*Bx + k32*By + k33*Bz
    Br_clean = Br.copy()

    limit = 5000   # 可以按需要改，比如 3000、5000、10000
    bad = (~np.isfinite(Br_clean)) | (np.abs(Br_clean) > limit)

    Br_clean[bad] = np.nan
    meta=B_map.meta.copy()
    meta.pop('BLANK',None)

    Br_map = sunpy.map.Map(Br_clean, meta)
    Br_map.save(os.path.join(hmi_path,'Br.fits'),overwrite=True)

1
2
3
4
5
6
7
8
